# MIXED-class error analysis

Compare BERTurk test predictions with no generated data and with generated data. The run names are kept in one configuration cell so they can be switched to Saturday's selected best-size run. Gold tokens come from `load_split_sentences("test")`; predictions are joined by `(doc_id, sent_id, tok_id)` and checked against the CSV token and gold columns. The error table is restricted to tokens where gold or either model uses `MIXED`.

In [ ]:
from pathlib import Path
import sys

REPO_ROOT = next(
    (parent for parent in [Path.cwd(), *Path.cwd().parents]
     if (parent / "src" / "lid_io.py").is_file()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError("Could not find src/lid_io.py from the notebook working directory")
if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))

# Update only these run names to compare a different selected model size.
RUNS = {
    "gen0": "berturk_gen0_seed1",
    "gen2500": "berturk_gen2500_seed1",
}
PREDICTION_DIR = REPO_ROOT / "results" / "predictions"
REPORT_PATH = REPO_ROOT / "results" / "error_analysis.md"
SPLIT = "test"

from lid_io import load_split_sentences
import pandas as pd
from IPython.display import display

print(f"Repository: {REPO_ROOT}")
print(f"Python: {sys.version.split()[0]}")
assert sys.version_info[:2] == (3, 12), "Run this notebook with Python 3.12"


## Load and align the real test split

The sentence loader supplies token order and gold labels. Enumerated token positions are checked against each prediction CSV's token and `lid` columns after an exact key join. This catches accidental split, tokenizer, or file mismatches instead of silently analyzing misaligned rows.

In [ ]:
gold_rows = []
for sentence in load_split_sentences(SPLIT):
    if len(sentence.tokens) != len(sentence.labels):
        raise ValueError(f"Token/label length mismatch for {(sentence.doc_id, sentence.sent_id)}")
    for token_idx, (token, label) in enumerate(zip(sentence.tokens, sentence.labels)):
        gold_rows.append({
            "doc_id": sentence.doc_id,
            "sent_id": int(sentence.sent_id),
            "tok_id": token_idx,
            "token": token,
            "gold": label,
        })
gold = pd.DataFrame(gold_rows)
key = ["doc_id", "sent_id", "tok_id"]
if gold.duplicated(key).any():
    raise ValueError("Gold test sentences contain duplicate token keys")

aligned = gold.copy()
for model, run_name in RUNS.items():
    path = PREDICTION_DIR / f"{run_name}_{SPLIT}.csv"
    if not path.is_file():
        raise FileNotFoundError(f"Prediction file not found: {path}")
    prediction = pd.read_csv(path, dtype={"doc_id": str, "token": str, "lid": str, "pred_lid": str})
    required = set(key + ["token", "lid", "pred_lid"])
    missing = required - set(prediction.columns)
    if missing:
        raise ValueError(f"{path.name} is missing columns: {sorted(missing)}")
    prediction["sent_id"] = prediction["sent_id"].astype(int)
    prediction["tok_id"] = prediction["tok_id"].astype(int)
    if prediction.duplicated(key).any():
        raise ValueError(f"{path.name} contains duplicate token keys")
    prediction = prediction[key + ["token", "lid", "pred_lid"]].rename(
        columns={"token": f"{model}_token", "lid": f"{model}_gold", "pred_lid": model}
    )
    aligned = aligned.merge(prediction, on=key, how="left", validate="one_to_one", indicator=f"{model}_match")
    unmatched = aligned[f"{model}_match"] != "both"
    if unmatched.any():
        raise ValueError(f"{path.name} does not cover {int(unmatched.sum())} gold test tokens")
    for column in (f"{model}_token", f"{model}_gold"):
        mismatch = aligned[column] != (aligned["token"] if column.endswith("_token") else aligned["gold"])
        if mismatch.any():
            row = aligned.loc[mismatch, key + ["token", "gold", column]].iloc[0].to_dict()
            raise ValueError(f"Gold/prediction mismatch in {path.name}: {row}")
    aligned = aligned.drop(columns=[f"{model}_token", f"{model}_gold", f"{model}_match"])

print(f"Aligned {len(aligned):,} test tokens from {len(load_split_sentences(SPLIT)):,} sentences")
display(aligned.head())


## MIXED metrics and prediction transitions

Precision, recall, and F1 use one-vs-rest `MIXED` decisions over the complete test split. Transition counts use exact full-label correctness on the MIXED-focused token set: fixed (gen0 wrong, gen2500 correct), degraded (gen0 correct, gen2500 wrong), persistently wrong, or correct in both.

In [ ]:
def mixed_metrics(frame, prediction_column):
    gold_mixed = frame["gold"].eq("MIXED")
    pred_mixed = frame[prediction_column].eq("MIXED")
    true_positive = int((gold_mixed & pred_mixed).sum())
    false_positive = int((~gold_mixed & pred_mixed).sum())
    false_negative = int((gold_mixed & ~pred_mixed).sum())
    precision = true_positive / (true_positive + false_positive) if true_positive + false_positive else 0.0
    recall = true_positive / (true_positive + false_negative) if true_positive + false_negative else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return {
        "model": prediction_column,
        "gold MIXED": int(gold_mixed.sum()),
        "predicted MIXED": int(pred_mixed.sum()),
        "TP": true_positive,
        "FP": false_positive,
        "FN": false_negative,
        "precision": precision,
        "recall": recall,
        "F1": f1,
    }

metrics = pd.DataFrame([mixed_metrics(aligned, model) for model in RUNS])
display(metrics.style.format({"precision": "{:.3f}", "recall": "{:.3f}", "F1": "{:.3f}"}))

candidate_mask = aligned[["gold", *RUNS.keys()]].eq("MIXED").any(axis=1)
candidates = aligned.loc[candidate_mask].copy()
candidates["gen0_correct"] = candidates["gen0"].eq(candidates["gold"])
candidates["gen2500_correct"] = candidates["gen2500"].eq(candidates["gold"])
def transition(row):
    if not row["gen0_correct"] and row["gen2500_correct"]:
        return "fixed"
    if row["gen0_correct"] and not row["gen2500_correct"]:
        return "degraded"
    if not row["gen0_correct"] and not row["gen2500_correct"]:
        return "persistently wrong"
    return "correct in both"

candidates["transition"] = candidates.apply(transition, axis=1)
transition_counts = candidates["transition"].value_counts().reindex(
    ["fixed", "degraded", "persistently wrong", "correct in both"], fill_value=0
).rename_axis("transition").reset_index(name="tokens")
display(transition_counts)
errors = candidates.loc[~(candidates["gen0_correct"] & candidates["gen2500_correct"])].copy()
print(f"MIXED-focused token instances: {len(candidates)}; error instances: {len(errors)}")


## Linguistic error review and report

Each error needs one canonical category. Review the displayed token, sentence, labels, and annotation metadata; add adjudicated cases to `MANUAL_REVIEW` keyed by `(doc_id, sent_id, tok_id)`. The default classifier uses observable form and labels for straightforward apostrophe, NE/MIXED, and diacritic-free cases; use manual entries for morphological judgments, gold-label concerns, and ambiguous cases. `other` is the explicit fallback. Notes are included verbatim in the report.

In [ ]:
CATEGORIES = [
    "apostrophe",
    "no apostrophe",
    "name vs MIXED",
    "no Turkish letters",
    "gold label looks wrong",
    "other",
]

# Linguistic adjudications belong here; keys are stable token coordinates.
# Example: ("post_001", 2, 4): ("no apostrophe", "English stem takes a Turkish suffix directly.")
MANUAL_REVIEW = {}

def default_category(row):
    predictions = (row["gen0"], row["gen2500"])
    if "NE" in (row["gold"], *predictions) and "MIXED" in (row["gold"], *predictions):
        return "name vs MIXED", "Named-entity and MIXED labels are confused for this token."
    if any(mark in row["token"] for mark in ("'", "’", "ʼ", "`")):
        return "apostrophe", "The token contains an apostrophe-like stem/suffix boundary."
    if row["gold"] == "MIXED" and row.get("borrowed_suffix") == "MIXED":
        return "no apostrophe", "Gold metadata marks a borrowed Turkish suffix, written without an apostrophe."
    if row["gold"] != "MIXED" and any(p == "MIXED" for p in predictions):
        return "other", "A model predicts MIXED, but the gold label is not MIXED; inspect the token in context."
    return "other", "The error does not fit another category without manual linguistic adjudication."

benchmark_path = REPO_ROOT / "data" / "benchmark.csv"
benchmark = pd.read_csv(benchmark_path, dtype={"doc_id": str}, keep_default_na=False)
metadata_columns = [column for column in ("borrowed_suffix", "ner") if column in benchmark.columns]
metadata = benchmark[key + metadata_columns].drop_duplicates(key)
errors = errors.merge(metadata, on=key, how="left", validate="one_to_one")
sentences_by_key = {}
for sentence in load_split_sentences(SPLIT):
    sentences_by_key[(sentence.doc_id, int(sentence.sent_id))] = " ".join(sentence.tokens)
errors["context"] = errors.apply(
    lambda row: sentences_by_key[(row["doc_id"], int(row["sent_id"]))], axis=1
)

reviewed = []
for _, row in errors.iterrows():
    token_key = (row["doc_id"], int(row["sent_id"]), int(row["tok_id"]))
    category, note = MANUAL_REVIEW.get(token_key, default_category(row))
    if category not in CATEGORIES:
        raise ValueError(f"Invalid category {category!r} for {token_key}")
    reviewed.append({**row.to_dict(), "category": category, "note": note})
error_table = pd.DataFrame(reviewed)
display(error_table[["doc_id", "sent_id", "tok_id", "token", "context", "gold", "gen0", "gen2500", "category", "note"]])

category_counts = error_table.groupby("category").agg(
    error_tokens=("token", "size"),
    gen0_errors=("gen0_correct", lambda values: int((~values).sum())),
    gen2500_errors=("gen2500_correct", lambda values: int((~values).sum())),
).reindex(CATEGORIES, fill_value=0).reset_index()
display(category_counts)


In [ ]:
def markdown_cell(value):
    return str(value).replace("|", "\\|").replace("\r", " ").replace("\n", " ")

def markdown_table(headers, rows):
    lines = ["| " + " | ".join(headers) + " |", "| " + " | ".join("---" for _ in headers) + " |"]
    lines.extend("| " + " | ".join(markdown_cell(value) for value in row) + " |" for row in rows)
    return "\n".join(lines)

metric_rows = [
    [row["model"], row["gold MIXED"], row["predicted MIXED"], row["TP"], row["FP"], row["FN"],
     f"{row['precision']:.3f}", f"{row['recall']:.3f}", f"{row['F1']:.3f}"]
    for row in metrics.to_dict(orient="records")
]
transition_rows = [[row["transition"], row["tokens"]] for row in transition_counts.to_dict(orient="records")]
category_rows = [
    [row["category"], row["error_tokens"], row["gen0_errors"], row["gen2500_errors"]]
    for row in category_counts.to_dict(orient="records")
]
detail_rows = []
for row in error_table.to_dict(orient="records"):
    detail_rows.append([
        row["token"], row["context"], row["gold"], row["gen0"], row["gen2500"], row["category"], row["note"]
    ])

fixed = int((candidates["transition"] == "fixed").sum())
degraded = int((candidates["transition"] == "degraded").sum())
persistent = int((candidates["transition"] == "persistently wrong").sum())
synthesis = (
    f"On the MIXED-focused set, adding synthetic data fixed {fixed} token(s), degraded {degraded}, "
    f"and left {persistent} persistently wrong. Inspect the category table and token-level notes below "
    "to distinguish successful suffix generalization on bare English stems from remaining apostrophe, "
    "named-entity, and annotation-related errors. The counts describe this seed and test split only."
)

report = "\n".join([
    "# BERTurk MIXED Error Analysis",
    "",
    f"Test split; runs: `{RUNS['gen0']}` vs `{RUNS['gen2500']}`. Gold and predictions aligned on `(doc_id, sent_id, tok_id)`.",
    "",
    "## MIXED metrics",
    "",
    markdown_table(["Model", "Gold MIXED", "Predicted MIXED", "TP", "FP", "FN", "Precision", "Recall", "F1"], metric_rows),
    "",
    "## Prediction transitions",
    "",
    markdown_table(["Transition", "Tokens"], transition_rows),
    "",
    "## Error counts by category",
    "",
    markdown_table(["Category", "Unique error tokens", f"{RUNS['gen0']} errors", f"{RUNS['gen2500']} errors"], category_rows),
    "",
    "## Token-level errors",
    "",
    markdown_table(["Token", "Sentence context", "Gold", "gen0", "gen2500", "Category", "Note"], detail_rows),
    "",
    "## Analysis",
    "",
    synthesis,
    "",
])
REPORT_PATH.parent.mkdir(parents=True, exist_ok=True)
REPORT_PATH.write_text(report, encoding="utf-8")
print(f"Wrote {REPORT_PATH} ({len(error_table)} token-level errors)")
